In [ ]:
import glob
import matplotlib.pyplot as plt
import numpy as np
import cv2


In [ ]:
# Input
folder_in_main = '/media/joris/rootfs/home/joris/data/kiwibes2026/20260706kiwibes/bes21-30-joris'


In [ ]:
def grid_spacing(img, visualize=False):
    
    # Blur
    blur = cv2.GaussianBlur(img, (15, 15), 0)  
    
    # Convert to grayscale
    gray = cv2.cvtColor(blur, cv2.COLOR_BGR2GRAY)
    
    if visualize:
        plt.figure(figsize=(20,10))
        plt.subplot(131)
        plt.imshow(img)
        plt.subplot(132)
        plt.imshow(blur)
        plt.subplot(133)
        plt.imshow(gray)
        plt.show()
    
    # Horizontal edges
    # Gradient in Y direction
    horizontal = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    
    # Vertical edges
    # Gradient in X direction
    vertical = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    
    # Convert to displayable images
    horizontal = cv2.convertScaleAbs(horizontal)
    vertical = cv2.convertScaleAbs(vertical)
    horiverti = horizontal.astype(np.bool) & vertical.astype(np.bool)
    
    if visualize:
        plt.figure(figsize=(20,10))
        plt.subplot(131)
        plt.imshow(horizontal)
        plt.subplot(132)
        plt.imshow(vertical)
        plt.subplot(133)
        plt.imshow(horiverti)
        plt.show()
    
    # Blur a few times
    horiverti_blur = horiverti.copy()
    for i in range(3):
        horiverti_blur = cv2.GaussianBlur(horiverti_blur.astype(np.uint8)*255, (15, 15), 0)  
    
    # Thresh
    horiverti_threshold = horiverti_blur > 200
    
    # Connected components
    analysis = cv2.connectedComponentsWithStats(horiverti_threshold.astype(np.uint8), 4, cv2.CV_32S)
    (totalLabels, label_ids, values, centroid) = analysis
    
    # Mask size filter
    mask_size_min = 500
    mask_size_max = 5000
    
    centroids_filtered = []
    label_ids_filtered = label_ids.copy()
    for i in range(totalLabels):
        mask_size = np.sum(label_ids == i)
        if mask_size >= mask_size_min and mask_size <= mask_size_max:
            centroids_filtered.append(centroid[i])
            pos_drop = np.array(np.where(label_ids_filtered == i)).transpose()
            for x, y in pos_drop:
                label_ids_filtered[x,y] = 0    
    centroids_filtered = np.array(centroids_filtered)
    
    if visualize:
        print(len(centroid), len(centroids_filtered))
        plt.figure(figsize=(20,10))
        plt.subplot(131)
        plt.imshow(horiverti_threshold)
        plt.subplot(132)
        plt.imshow(label_ids)
        plt.scatter([c[0] for c in centroid], [c[1] for c in centroid], s = 100, marker='o',  facecolors='none', edgecolors='red')
        plt.subplot(133)
        plt.imshow(label_ids_filtered)
        plt.scatter([c[0] for c in centroids_filtered], [c[1] for c in centroids_filtered],
                    s = 100, marker='o',  facecolors='none', edgecolors='red')
        plt.show()
    
    # Distances to nearest neighbour
    detections = centroids_filtered
    distances = []
    for i in range(len(detections)):
        lowest_distance = np.inf
        for j in range(len(detections)):
            if not j == i:
                distance = np.linalg.norm(detections[i] - detections[j])
                lowest_distance = min(lowest_distance, distance)
        distances.append(lowest_distance)
    
    # Keep middle section
    distances.sort()
    distances = distances[len(distances)//5:4*len(distances)//5]
    
    # Grid spacing is the mean 
    grid_spacing = np.mean(distances)
    
    return grid_spacing



# img = cv2.imread(file_in)
# grid_spacing(img, visualize=False)

In [ ]:
files_in = np.random.choice(glob.glob(f'{folder_in_main}/*/*'), 10)
imgs = []
grid_spacings = []
for file_in in files_in:
    img = cv2.imread(file_in)
    imgs.append(img)
    grid_spacings.append(grid_spacing(img, visualize=False))

In [ ]:
grid_spacings

In [ ]:
for i in np.argsort(grid_spacings):
    plt.figure()
    plt.imshow(imgs[i])
    plt.title(f'grid_spcaing: {grid_spacings[i]}')
    plt.show()

# EXP

In [ ]:
STOP

In [ ]:
# A grid proposal
x0 = 5
y0 = 8
s = 20 # Grid spacing (pix)
a = 30*np.pi/180 # Grid orientation, radians

# Limits for generation of cross points
xmin = np.min(detections[:,0]) - 2*s
xmax = np.min(detections[:,0]) + 2*s
ymin = np.min(detections[:,1]) - 2*s
ymax = np.min(detections[:,1]) + 2*s

# Jump vectors
hori_jump = np.array([s*np.cos(a), s*np.sin(a)])
verti_jump = np.array([s*np.cos(a+np.pi/2), s*np.sin(a+np.pi/2)])

# # Start
# cross_point = np.array([x0, y0])
# cross_points = [cross_point]
# print(cross_points)

# # Add

# # Try hori move
# cross_point = cross_points[-1] + hori_jump

# # Check if inside 
# if cross_point[0] >= xmin and cross_point[0] <= xmax and cross_point[1] >= ymin and cross_point[1] <= ymax:
#     cross_points.append(cross_point)

# else:
#     # Try verti jump
#     cross_point = cross_points[-1] + hori_jump

In [ ]:
# Distances to nearest neighbour
distances = []
for i in range(len(detections)):
    lowest_distance = np.inf
    for j in range(len(detections)):
        if not j == i:
            distance = np.linalg.norm(detections[i] - detections[j])
            lowest_distance = min(lowest_distance, distance)
    distances.append(lowest_distance)

# plt.hist(distances)

# Keep middle section
distances.sort()
distances = distances[len(distances)//5:4*len(distances)//5]

# plt.hist(distances)

# Grid spacing is the mean 
grid_spacing = np.mean(distances)

In [ ]:
test = gray[0:500,0:500]
plt.imshow(test)
plt.scatter([60, 60+184], [110, 110], color = 'red')

# Fourier

In [ ]:
test = gray#[0:1500,:]

dft = cv2.dft(np.float32(test),flags = cv2.DFT_COMPLEX_OUTPUT)
dft_shift = np.fft.fftshift(dft)

magnitude_spectrum = 20*np.log(cv2.magnitude(dft_shift[:,:,0],dft_shift[:,:,1]))

plt.subplot(121),plt.imshow(test, cmap = 'gray')
plt.title('Input Image'), plt.xticks([]), plt.yticks([])
plt.subplot(122),plt.imshow(magnitude_spectrum, cmap = 'gray')
plt.title('Magnitude Spectrum'), plt.xticks([]), plt.yticks([])
plt.show()

In [ ]:
plt.imshow(magnitude_spectrum)

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Convert image to float32
img = np.float32(test)

# DFT
dft = cv2.dft(img, flags=cv2.DFT_COMPLEX_OUTPUT)
dft_shift = np.fft.fftshift(dft)

# Magnitude spectrum
magnitude = cv2.magnitude(
    dft_shift[:, :, 0],
    dft_shift[:, :, 1]
)

# Log spectrum for visualization
magnitude_spectrum = 20 * np.log(magnitude + 1e-10)

# Image dimensions
Ny, Nx = img.shape[:2]

# Frequency coordinates
fx = np.fft.fftshift(np.fft.fftfreq(Nx))
fy = np.fft.fftshift(np.fft.fftfreq(Ny))

FX, FY = np.meshgrid(fx, fy)

# Remove DC component (center)
cy, cx = Ny // 2, Nx // 2

magnitude_no_dc = magnitude.copy()
magnitude_no_dc[cy, cx] = 0

# Optional: ignore very low frequencies
# This prevents slow illumination variations
# from being selected as the fundamental.
radius = np.sqrt((FX / (1 / Nx))**2 + (FY / (1 / Ny))**2)

magnitude_no_dc[radius < 2] = 0

# Find strongest frequency peak
peak_y, peak_x = np.unravel_index(
    np.argmax(magnitude_no_dc),
    magnitude_no_dc.shape
)

# Frequency at peak
f_x = FX[peak_y, peak_x]
f_y = FY[peak_y, peak_x]

# Spatial frequency magnitude
f = np.sqrt(f_x**2 + f_y**2)

# Wavelength in pixels
wavelength = 1 / f

print("Peak frequency fx:", f_x)
print("Peak frequency fy:", f_y)
print("Spatial frequency:", f)
print("Fundamental wavelength:", wavelength, "pixels")

# Plot
plt.figure(figsize=(12, 5))

plt.subplot(121)
plt.imshow(test, cmap="gray")
plt.title("Input Image")
plt.axis("off")

plt.subplot(122)
plt.imshow(magnitude_spectrum, cmap="gray")
plt.scatter(
    peak_x,
    peak_y,
    color="red",
    s=50,
    marker="x"
)
plt.title(f"Magnitude Spectrum\nWavelength = {wavelength:.2f} pixels")
plt.axis("off")

plt.show()